In [4]:
"""
Model Quantization and Conversion - FIXED VERSION
Convert Keras model to TensorFlow Lite INT8 format for microcontroller deployment
"""

import numpy as np
import tensorflow as tf
from tensorflow import keras
import pickle
from pathlib import Path
import os

print("="*60)
print("MODEL QUANTIZATION TO TENSORFLOW LITE")
print("="*60)
print(f"TensorFlow version: {tf.__version__}")

# ============================================================================
# STEP 1: Load Trained Model and Data
# ============================================================================
print("\n" + "="*60)
print("STEP 1: Loading Model and Data")
print("="*60)

# Load IDCNN model
model_path = Path('../models/activity_model.h5')
model = keras.models.load_model(model_path)
print(f"✓ Loaded IDCNN model from: {model_path}")

# Load preprocessing info
with open('../data/processed/preprocessing_info.pkl', 'rb') as f:
    preprocessing_info = pickle.load(f)

# Load test data for accuracy comparison
X_test = np.load('../data/processed/X_test.npy')
y_test = np.load('../data/processed/y_test.npy')

# Load training data for representative dataset
X_train = np.load('../data/processed/X_train.npy')

print(f"✓ Loaded test data: {X_test.shape}")
print(f"✓ Loaded training data: {X_train.shape}")

# ============================================================================
# STEP 2: Evaluate Original Model
# ============================================================================
print("\n" + "="*60)
print("STEP 2: Original Model Performance")
print("="*60)

# Evaluate original model
loss_original, acc_original = model.evaluate(X_test, y_test, verbose=0)
print(f"Original Model:")
print(f"  Accuracy: {acc_original*100:.2f}%")
print(f"  Loss: {loss_original:.4f}")

# Get model size
model_size_original = os.path.getsize(model_path) / 1024  # KB
print(f"  File size: {model_size_original:.2f} KB")

# ============================================================================
# STEP 3: Create Representative Dataset
# ============================================================================
print("\n" + "="*60)
print("STEP 3: Creating Representative Dataset")
print("="*60)

def representative_dataset_gen():
    """
    Generator for representative dataset used in quantization
    This helps TFLite determine optimal quantization parameters
    """
    # Use 100 random samples from training data
    num_samples = min(100, len(X_train))
    indices = np.random.choice(len(X_train), num_samples, replace=False)
    
    for i in indices:
        # TFLite expects float32 input
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

print("✓ Representative dataset generator created")
print(f"  Using {min(100, len(X_train))} samples from training data")

# ============================================================================
# STEP 4: Convert to TensorFlow Lite (INT8 Quantization)
# ============================================================================
print("\n" + "="*60)
print("STEP 4: Converting to TensorFlow Lite (INT8)")
print("="*60)

# Create TFLite converter
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Set optimization flags for full integer quantization
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Provide representative dataset
converter.representative_dataset = representative_dataset_gen

# Ensure all ops are integer (INT8)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set input/output types to INT8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

print("Quantization settings:")
print("  Optimization: DEFAULT (full integer quantization)")
print("  Target ops: TFLITE_BUILTINS_INT8")
print("  Input type: INT8")
print("  Output type: INT8")

# Convert model
print("\nConverting model (this may take 30-60 seconds)...")
tflite_model = converter.convert()

print("✓ Model converted to TensorFlow Lite!")

# Save TFLite model
tflite_path = Path('../models/activity_model_quantized.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

tflite_size = len(tflite_model) / 1024  # KB
print(f"✓ Saved: {tflite_path}")
print(f"  File size: {tflite_size:.2f} KB")

# Calculate size reduction
size_reduction = ((model_size_original - tflite_size) / model_size_original) * 100
print(f"  Size reduction: {size_reduction:.1f}%")

# ============================================================================
# STEP 5: Test Quantized Model Accuracy
# ============================================================================
print("\n" + "="*60)
print("STEP 5: Testing Quantized Model")
print("="*60)

# Load TFLite model
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite Model Details:")
print(f"  Input shape: {input_details[0]['shape']}")
print(f"  Input type: {input_details[0]['dtype']}")
print(f"  Output shape: {output_details[0]['shape']}")
print(f"  Output type: {output_details[0]['dtype']}")

# Get quantization parameters
input_scale = input_details[0]['quantization'][0]
input_zero_point = input_details[0]['quantization'][1]
output_scale = output_details[0]['quantization'][0]
output_zero_point = output_details[0]['quantization'][1]

print(f"\nQuantization Parameters:")
print(f"  Input scale: {input_scale:.6f}, zero_point: {input_zero_point}")
print(f"  Output scale: {output_scale:.6f}, zero_point: {output_zero_point}")

# Test on all test samples
print("\nRunning inference on test set...")
predictions = []

for i in range(len(X_test)):
    # Quantize input
    input_data = X_test[i:i+1].astype(np.float32)
    input_data_quantized = (input_data / input_scale + input_zero_point).astype(np.int8)
    
    # Run inference
    interpreter.set_tensor(input_details[0]['index'], input_data_quantized)
    interpreter.invoke()
    
    # Get output
    output_data = interpreter.get_tensor(output_details[0]['index'])
    
    # Dequantize output
    output_dequantized = (output_data.astype(np.float32) - output_zero_point) * output_scale
    
    # Get prediction
    pred = np.argmax(output_dequantized)
    predictions.append(pred)

predictions = np.array(predictions)

# Calculate accuracy
acc_quantized = np.mean(predictions == y_test)
print(f"✓ Inference complete")

print(f"\nQuantized Model:")
print(f"  Accuracy: {acc_quantized*100:.2f}%")

# Calculate accuracy drop
acc_drop = (acc_original - acc_quantized) * 100
print(f"  Accuracy drop: {acc_drop:.2f}%")

# ============================================================================
# STEP 6: Comparison Summary
# ============================================================================
print("\n" + "="*60)
print("STEP 6: Comparison Summary")
print("="*60)

comparison_table = f"""
{'Metric':<25} {'Original Model':<20} {'Quantized Model':<20} {'Change':<15}
{'-'*80}
{'Format':<25} {'Keras (.h5)':<20} {'TFLite (.tflite)':<20} {'-':<15}
{'File Size':<25} {f'{model_size_original:.2f} KB':<20} {f'{tflite_size:.2f} KB':<20} {f'-{size_reduction:.1f}%':<15}
{'Test Accuracy':<25} {f'{acc_original*100:.2f}%':<20} {f'{acc_quantized*100:.2f}%':<20} {f'-{acc_drop:.2f}%':<15}
{'Weights Precision':<25} {'32-bit float':<20} {'8-bit int':<20} {'4x smaller':<15}
"""

print(comparison_table)

# ============================================================================
# STEP 7: Convert to C Header File
# ============================================================================
print("\n" + "="*60)
print("STEP 7: Converting to C Header File")
print("="*60)

# Read TFLite model as bytes
with open(tflite_path, 'rb') as f:
    tflite_bytes = f.read()

# Convert to C array
def create_c_array(data, var_name):
    """Convert binary data to C array format"""
    c_array = f"// Auto-generated file\n"
    c_array += f"// TensorFlow Lite model for activity recognition\n"
    c_array += f"// Model size: {len(data)} bytes\n\n"
    c_array += f"alignas(8) const unsigned char {var_name}[] = {{\n"
    
    # Write bytes in rows of 12
    for i in range(0, len(data), 12):
        row = data[i:i+12]
        c_array += "  " + ", ".join([f"0x{b:02x}" for b in row])
        if i + 12 < len(data):
            c_array += ","
        c_array += "\n"
    
    c_array += f"}};\n"
    c_array += f"const unsigned int {var_name}_len = {len(data)};\n"
    
    return c_array

# Create C header file
c_header = create_c_array(tflite_bytes, "activity_model_data")

# Save C header
c_header_path = Path('../models/model_data.h')
with open(c_header_path, 'w') as f:
    f.write(c_header)

print(f"✓ Saved: {c_header_path}")
print(f"  Array name: activity_model_data")
print(f"  Array size: {len(tflite_bytes)} bytes")

# ============================================================================
# STEP 8: Create Config Header for C++ (FIXED)
# ============================================================================
print("\n" + "="*60)
print("STEP 8: Creating C++ Config Header (FIXED)")
print("="*60)

# Get values - handle missing keys gracefully
num_classes = len(preprocessing_info['label_to_activity'])
num_features = len(preprocessing_info['mean'])  # Infer from mean array

# Try to get from config, fallback to defaults
if 'config' in preprocessing_info and isinstance(preprocessing_info['config'], dict):
    window_size = preprocessing_info['config'].get('window_size', 128)
    sampling_rate = preprocessing_info['config'].get('sampling_rate', 50)
else:
    # Fallback values
    window_size = 128
    sampling_rate = 50

print(f"Configuration values:")
print(f"  NUM_CLASSES: {num_classes}")
print(f"  NUM_FEATURES: {num_features}")
print(f"  WINDOW_SIZE: {window_size}")
print(f"  SAMPLING_RATE: {sampling_rate}")

# Create config.h for C++ firmware
config_content = f"""// Auto-generated configuration file
// Generated from Python preprocessing

#ifndef CONFIG_H
#define CONFIG_H

// Model configuration
#define NUM_CLASSES {num_classes}
#define NUM_FEATURES {num_features}
#define WINDOW_SIZE {window_size}
#define SAMPLING_RATE {sampling_rate}

// Normalization parameters (from training data)
const float SENSOR_MEAN[NUM_FEATURES] = {{
    {', '.join([f'{m:.6f}f' for m in preprocessing_info['mean']])}
}};

const float SENSOR_STD[NUM_FEATURES] = {{
    {', '.join([f'{s:.6f}f' for s in preprocessing_info['std']])}
}};

// Quantization parameters (from TFLite model)
const float INPUT_SCALE = {input_scale:.8f}f;
const int8_t INPUT_ZERO_POINT = {input_zero_point};
const float OUTPUT_SCALE = {output_scale:.8f}f;
const int8_t OUTPUT_ZERO_POINT = {output_zero_point};

// Activity labels
const char* ACTIVITY_LABELS[NUM_CLASSES] = {{
    {', '.join([f'"{preprocessing_info["label_to_activity"][i]}"' for i in range(num_classes)])}
}};

// Sensor column order (for reference)
// 0: Ax, 1: Ay, 2: Az, 3: Gx, 4: Gy, 5: Gz

#endif // CONFIG_H
"""

config_path = Path('../models/config.h')
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"\n✓ Saved: {config_path}")
print("  Contains: normalization params, quantization params, labels")

# Verify file was created
if config_path.exists():
    print(f"✅ config.h successfully created!")
    print(f"   File size: {config_path.stat().st_size} bytes")
else:
    print("❌ Error creating config.h")

# ============================================================================
# STEP 9: Save Quantization Report
# ============================================================================
print("\n" + "="*60)
print("STEP 9: Saving Quantization Report")
print("="*60)

quantization_report = {
    'original_model': {
        'accuracy': float(acc_original),
        'loss': float(loss_original),
        'size_kb': float(model_size_original),
        'format': 'Keras .h5',
        'precision': 'float32'
    },
    'quantized_model': {
        'accuracy': float(acc_quantized),
        'size_kb': float(tflite_size),
        'format': 'TFLite INT8',
        'precision': 'int8'
    },
    'changes': {
        'accuracy_drop_percent': float(acc_drop),
        'size_reduction_percent': float(size_reduction),
        'compression_ratio': float(model_size_original / tflite_size)
    },
    'quantization_params': {
        'input_scale': float(input_scale),
        'input_zero_point': int(input_zero_point),
        'output_scale': float(output_scale),
        'output_zero_point': int(output_zero_point)
    },
    'config': {
        'num_classes': num_classes,
        'num_features': num_features,
        'window_size': window_size,
        'sampling_rate': sampling_rate
    }
}

with open('../models/quantization_report.pkl', 'wb') as f:
    pickle.dump(quantization_report, f)

print("✓ Saved: ../models/quantization_report.pkl")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*60)
print("QUANTIZATION COMPLETE!")
print("="*60)

print(f"""
Results:
  ✓ Original model: {acc_original*100:.2f}% accuracy, {model_size_original:.1f} KB
  ✓ Quantized model: {acc_quantized*100:.2f}% accuracy, {tflite_size:.1f} KB
  ✓ Size reduction: {size_reduction:.1f}% ({model_size_original/tflite_size:.1f}x smaller)
  ✓ Accuracy drop: {acc_drop:.2f}% (acceptable for embedded deployment)

Files Created:
  ✓ models/activity_model_quantized.tflite (quantized model)
  ✓ models/model_data.h (C array for Arduino)
  ✓ models/config.h (C++ configuration header)
  ✓ models/quantization_report.pkl (metadata)

Ready for Deployment:
  ✅ model_data.h → Copy to C++ project
  ✅ config.h → Copy to C++ project
  
Next Steps:
  1. Copy model_data.h and config.h to your C++ project
  2. Set up XIAO nRF52840 hardware
  3. Implement firmware for real-time inference
  
Status: ✅ READY FOR EMBEDDED DEPLOYMENT
""")

MODEL QUANTIZATION TO TENSORFLOW LITE
TensorFlow version: 2.13.0

STEP 1: Loading Model and Data
✓ Loaded IDCNN model from: ../models/activity_model.h5
✓ Loaded test data: (413, 128, 6)
✓ Loaded training data: (2478, 128, 6)

STEP 2: Original Model Performance


2025-12-08 14:53:26.064891: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Original Model:
  Accuracy: 99.52%
  Loss: 0.0122
  File size: 363.70 KB

STEP 3: Creating Representative Dataset
✓ Representative dataset generator created
  Using 100 samples from training data

STEP 4: Converting to TensorFlow Lite (INT8)
Quantization settings:
  Optimization: DEFAULT (full integer quantization)
  Target ops: TFLITE_BUILTINS_INT8
  Input type: INT8
  Output type: INT8

Converting model (this may take 30-60 seconds)...
INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpifbtw5of/assets


INFO:tensorflow:Assets written to: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpifbtw5of/assets


✓ Model converted to TensorFlow Lite!
✓ Saved: ../models/activity_model_quantized.tflite
  File size: 38.67 KB
  Size reduction: 89.4%

STEP 5: Testing Quantized Model
TFLite Model Details:
  Input shape: [  1 128   6]
  Input type: <class 'numpy.int8'>
  Output shape: [1 5]
  Output type: <class 'numpy.int8'>

Quantization Parameters:
  Input scale: 0.149977, zero_point: 7
  Output scale: 0.003906, zero_point: -128

Running inference on test set...
✓ Inference complete

Quantized Model:
  Accuracy: 99.03%
  Accuracy drop: 0.48%

STEP 6: Comparison Summary

Metric                    Original Model       Quantized Model      Change         
--------------------------------------------------------------------------------
Format                    Keras (.h5)          TFLite (.tflite)     -              
File Size                 363.70 KB            38.67 KB             -89.4%         
Test Accuracy             99.52%               99.03%               -0.48%         
Weights Precision  

/opt/homebrew/Caskroom/miniforge/base/envs/tinyml/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:887: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2025-12-08 14:53:27.484731: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
2025-12-08 14:53:27.485002: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2025-12-08 14:53:27.485572: I tensorflow/cc/saved_model/reader.cc:45] Reading SavedModel from: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpifbtw5of
2025-12-08 14:53:27.487043: I tensorflow/cc/saved_model/reader.cc:91] Reading meta graph with tags { serve }
2025-12-08 14:53:27.487047: I tensorflow/cc/saved_model/reader.cc:132] Reading SavedModel debug info (if present) from: /var/folders/52/bpf46hx95sl2yr7m61mcmnlw0000gn/T/tmpifbtw5of
2025-12-08 14:53:27.492722: I tensorflow/cc/saved_model/loader.